In [1]:
import duckdb

con = duckdb.connect('database.duckdb')


In [2]:
con = duckdb.connect('database.duckdb')

# atas_2025.csv
# contratacoes_2025.csv
# notas_empenho_2025.csv
# servidores_2025.csv
# liquidacoes_2025.csv
# contratos_2025.csv
con.execute("""
    CREATE TABLE atas AS 
    SELECT * FROM read_csv_auto(
        './dados/atas_*.csv', 
        union_by_name=true
    )
""")
con.execute("""
    CREATE TABLE contratacoes AS 
    SELECT * FROM read_csv_auto(
        './dados/contratacoes_*.csv', 
        union_by_name=true
    )
""")

con.execute("""
    CREATE TABLE servidores AS 
    SELECT * FROM read_csv_auto(
        './dados/servidores_*.csv', 
        union_by_name=true
    )
""")
con.execute("""
    CREATE TABLE empenhos AS 
    SELECT * FROM read_csv_auto(
        './dados/notas_empenho_*.csv', 
        union_by_name=true
    )
""")

con.execute("""
CREATE TABLE liquidacoes AS
SELECT * FROM read_csv_auto(
    './dados/liquidacoes_*.csv',
    union_by_name=true
)
""")

con.execute("""
CREATE TABLE contratos AS
SELECT * FROM read_csv_auto(
    './dados/contratos_*.csv',
    union_by_name=true
)
""")


total_empenhos = con.execute("SELECT COUNT(*) FROM empenhos").fetchone()
print(f"Total de registros de todos os anos: {total_empenhos[0]}")

tabelas = con.execute("SHOW TABLES").fetchall()

for (nome_tabela,) in tabelas:
    schema = con.execute(f"DESCRIBE {nome_tabela}").fetchall()
    print(f"\nTabela: {nome_tabela}")
    for coluna in schema:
        print(f"  {coluna[0]} ({coluna[1]})")

    qtd = con.execute(
        f"SELECT COUNT(*) FROM {nome_tabela}"
    ).fetchone()[0]

    print(f"{nome_tabela}: {qtd:,} registros")
    
con.close()



Total de registros de todos os anos: 328248
atas: 2,500 registros
contratacoes: 1,200 registros
contratos: 7,480 registros
empenhos: 328,248 registros
liquidacoes: 381,176 registros
servidores: 1,200 registros


In [7]:
import duckdb

con = duckdb.connect('database.duckdb')

total_empenhos = con.execute("SELECT COUNT(*) FROM empenhos").fetchone()
print(f"Total de registros de todos os anos: {total_empenhos[0]}")

tabelas = con.execute("SHOW TABLES").fetchall()

for (nome_tabela,) in tabelas:
    schema = con.execute(f"DESCRIBE {nome_tabela}").fetchall()
    print(f"\nTabela: {nome_tabela}")
    for coluna in schema:
        print(f"  {coluna[0]} ({coluna[1]})")

    qtd = con.execute(
        f"SELECT COUNT(*) FROM {nome_tabela}"
    ).fetchone()[0]

    print(f"{nome_tabela}: {qtd:,} registros")
    
con.close()

Total de registros de todos os anos: 328248

Tabela: atas
  ano (BIGINT)
  mes (BIGINT)
  numeroProcesso (VARCHAR)
  nomeOrgao (VARCHAR)
  objeto (VARCHAR)
  tipoProcesso (VARCHAR)
  modalidade (VARCHAR)
  tipoJulgamento (VARCHAR)
  situacao (VARCHAR)
  dataCriacao (DATE)
  valorLicitado (DOUBLE)
  linkDocumentoAta (VARCHAR)
atas: 2,500 registros

Tabela: contratacoes
  numeroProcesso (VARCHAR)
  numeroLicitacao (VARCHAR)
  cadastroCge (VARCHAR)
  tipoDocumento (VARCHAR)
  nomeOrgao (VARCHAR)
  objeto (VARCHAR)
  modalidade (VARCHAR)
  tipoLicitacao (VARCHAR)
  criterioClassificacao (VARCHAR)
  situacao (VARCHAR)
  dataCriacao (DATE)
  dataAbertura (DATE)
  dataAdjudicacao (DATE)
  valorEstimado (DOUBLE)
  valorAdjudicado (DOUBLE)
  numParticipantes (DOUBLE)
  registroPreco (VARCHAR)
  amparoLegal (VARCHAR)
  urlEdital (VARCHAR)
  urlContratacao (VARCHAR)
  urlPncp (VARCHAR)
  documentos (VARCHAR)
  participantes (VARCHAR)
contratacoes: 1,200 registros

Tabela: contratos
  anoInicioVig

In [9]:
import duckdb

con = duckdb.connect('database.duckdb')
con.execute("ALTER TABLE empenhos ADD COLUMN IF NOT EXISTS valor DOUBLE")
con.execute("UPDATE empenhos SET valor = valorEmpenhado WHERE valor IS NULL")
con.close()

In [11]:
import duckdb

con = duckdb.connect('database.duckdb')
print(con.execute('DESCRIBE empenhos').fetchdf().to_string(index=False))
con.close()

             column_name column_type null  key default extra
                     ano      BIGINT  YES None    None  None
                     mes      BIGINT  YES None    None  None
               tipoPoder     VARCHAR  YES None    None  None
             codigoOrgao      BIGINT  YES None    None  None
               nomeOrgao     VARCHAR  YES None    None  None
           codigoUnidade     VARCHAR  YES None    None  None
           numeroEmpenho      BIGINT  YES None    None  None
             dataEmpenho        DATE  YES None    None  None
            numeroEmenda     VARCHAR  YES None    None  None
         tipoNotaEmpenho      BIGINT  YES None    None  None
           descricaoTipo     VARCHAR  YES None    None  None
        descricaoEmpenho     VARCHAR  YES None    None  None
       codigoTipoCredito      BIGINT  YES None    None  None
             tipoCredito     VARCHAR  YES None    None  None
               dataSaida        DATE  YES None    None  None
             dataRetorno

In [12]:
import duckdb

con = duckdb.connect('database.duckdb')
con.execute("ALTER TABLE empenhos ADD COLUMN IF NOT EXISTS orgao VARCHAR")
con.execute("ALTER TABLE empenhos ADD COLUMN IF NOT EXISTS data_empenho DATE")
con.execute("ALTER TABLE empenhos ADD COLUMN IF NOT EXISTS tipo VARCHAR")
con.execute("UPDATE empenhos SET orgao = nomeOrgao WHERE orgao IS NULL")
con.execute("UPDATE empenhos SET data_empenho = dataEmpenho WHERE data_empenho IS NULL")
con.execute("UPDATE empenhos SET tipo = descricaoTipo WHERE tipo IS NULL")
con.close()

In [14]:
import plotly.express as px

_sunburst = px.sunburst

def sunburst_por_tipo_e_orgao(data_frame, **kwargs):
    kwargs.pop('labels', None)
    kwargs.pop('parents', None)
    kwargs.pop('values', None)
    return _sunburst(
        data_frame,
        path=['nivel_1', 'nivel_2'],
        values='valor_total',
        **kwargs,
    )

px.sunburst = sunburst_por_tipo_e_orgao

In [15]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ==================== CONFIGURAÇÃO ====================
con = duckdb.connect('database.duckdb')

# ==================== 1. DASHBOARD EXECUTIVO ====================
print("Gerando dashboard executivo...")

# Dados para o dashboard
kpis = con.execute("""
    SELECT 
        COUNT(*) as total_empenhos,
        SUM(valor) as valor_total,
        ROUND(AVG(valor), 2) as valor_medio,
        COUNT(DISTINCT orgao) as total_orgaos,
        MAX(data_empenho) as ultima_atualizacao
    FROM empenhos
""").fetchdf()

# Criar figura com subplots
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "Evolução Temporal", "Top 10 Órgãos",
        "Distribuição por Tipo", "Quartis de Valores",
        "Empenhos por Mês", "Concentração (Pareto)"
    ),
    specs=[
        [{"type": "scatter"}, {"type": "bar"}],
        [{"type": "pie"}, {"type": "box"}],
        [{"type": "bar"}, {"type": "scatter"}]
    ]
)

# 1.1 Evolução Temporal
temporal = con.execute("""
    SELECT 
        DATE_TRUNC('month', data_empenho) as periodo,
        SUM(valor) as valor_total
    FROM empenhos
    WHERE data_empenho IS NOT NULL
    GROUP BY DATE_TRUNC('month', data_empenho)
    ORDER BY periodo
""").fetchdf()

fig.add_trace(
    go.Scatter(
        x=temporal['periodo'],
        y=temporal['valor_total']/1e6,
        mode='lines+markers',
        name='Valor Total',
        line=dict(color='#2E86AB', width=3),
        marker=dict(size=5)
    ),
    row=1, col=1
)

# 1.2 Top 10 Órgãos
top_orgaos = con.execute("""
    SELECT 
        orgao,
        SUM(valor) as valor_total
    FROM empenhos
    GROUP BY orgao
    ORDER BY valor_total DESC
    LIMIT 10
""").fetchdf()

fig.add_trace(
    go.Bar(
        y=top_orgaos['orgao'],
        x=top_orgaos['valor_total']/1e6,
        orientation='h',
        name='Órgãos',
        marker=dict(color='#A23B72')
    ),
    row=1, col=2
)

# 1.3 Distribuição por Tipo
dist_tipo = con.execute("""
    SELECT 
        tipo,
        COUNT(*) as quantidade
    FROM empenhos
    GROUP BY tipo
    ORDER BY quantidade DESC
""").fetchdf()

fig.add_trace(
    go.Pie(
        labels=dist_tipo['tipo'],
        values=dist_tipo['quantidade'],
        name='Tipo'
    ),
    row=2, col=1
)

# 1.4 Box plot dos valores
valores = con.execute("SELECT valor FROM empenhos WHERE valor > 0").fetchall()
fig.add_trace(
    go.Box(
        y=[v[0] for v in valores],
        name='Distribuição',
        marker=dict(color='#F18F01')
    ),
    row=2, col=2
)

# 1.5 Empenhos por Mês
empenhos_mes = con.execute("""
    SELECT 
        MONTH(data_empenho) as mes,
        COUNT(*) as quantidade
    FROM empenhos
    WHERE MONTH(data_empenho) IS NOT NULL
    GROUP BY MONTH(data_empenho)
    ORDER BY mes
""").fetchdf()

mes_nomes = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 
             'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

fig.add_trace(
    go.Bar(
        x=[mes_nomes[int(m)-1] for m in empenhos_mes['mes']],
        y=empenhos_mes['quantidade'],
        name='Quantidade',
        marker=dict(color='#06A77D')
    ),
    row=3, col=1
)

# 1.6 Pareto (Concentração)
pareto = con.execute("""
    WITH ranking AS (
        SELECT 
            ROW_NUMBER() OVER (ORDER BY SUM(valor) DESC) as rank,
            orgao,
            SUM(valor) as valor_total,
            SUM(SUM(valor)) OVER (ORDER BY SUM(valor) DESC) as acumulado,
            SUM(SUM(valor)) OVER () as total
        FROM empenhos
        GROUP BY orgao
    )
    SELECT 
        rank,
        100.0 * acumulado / total as percentual_acumulado
    FROM ranking
    LIMIT 30
""").fetchdf()

fig.add_trace(
    go.Scatter(
        x=pareto['rank'],
        y=pareto['percentual_acumulado'],
        mode='lines+markers',
        name='Acumulado %',
        line=dict(color='#D62828', width=3),
        marker=dict(size=6)
    ),
    row=3, col=2
)

# Atualizar layout
fig.update_xaxes(title_text="Data", row=1, col=1)
fig.update_yaxes(title_text="Valor (Milhões R$)", row=1, col=1)
fig.update_xaxes(title_text="Valor (Milhões R$)", row=1, col=2)
fig.update_xaxes(title_text="Quartis", row=2, col=2)
fig.update_yaxes(title_text="Valores (log scale)", type="log", row=2, col=2)
fig.update_xaxes(title_text="Mês", row=3, col=1)
fig.update_xaxes(title_text="Rank de Órgão", row=3, col=2)
fig.update_yaxes(title_text="Acumulado %", row=3, col=2)

fig.update_layout(
    height=1200,
    width=1600,
    title_text="📊 Dashboard Executivo - Empenhos",
    showlegend=False,
    template="plotly_white"
)

fig.write_html("dashboard_empenhos.html")
print("✅ Dashboard salvo: dashboard_empenhos.html")


# ==================== 2. VISUALIZAÇÃO SUNBURST (Hierarquia) ====================
print("\nGerando visualização em Sunburst...")

hierarchia = con.execute("""
    SELECT 
        'Total' as nivel_0,
        COALESCE(tipo, 'Sem Tipo') as nivel_1,
        COALESCE(orgao, 'Sem Órgão') as nivel_2,
        COUNT(*) as quantidade,
        SUM(valor) as valor_total
    FROM empenhos
    GROUP BY tipo, orgao
    ORDER BY valor_total DESC
    LIMIT 50
""").fetchdf()

fig_sunburst = px.sunburst(
    hierarchia,
    labels=['Total'] + hierarchia['nivel_1'].tolist() + hierarchia['nivel_2'].tolist(),
    parents=[''] + ['Total'] * hierarchia['nivel_1'].nunique() + hierarchia['nivel_1'].tolist(),
    values=[hierarchia['valor_total'].sum()] + hierarchia['valor_total'].tolist(),
    title="🌞 Hierarquia: Tipo → Órgão",
    color='valor_total',
    color_continuous_scale='Viridis',
)

fig_sunburst.write_html("sunburst_empenhos.html")
print("✅ Sunburst salvo: sunburst_empenhos.html")


# ==================== 3. TREEMAP (Proporção Visual) ====================
print("\nGerando Treemap...")

treemap_data = con.execute("""
    SELECT 
        orgao,
        COUNT(*) as quantidade,
        SUM(valor) as valor_total,
        ROUND(100.0 * SUM(valor) / (SELECT SUM(valor) FROM empenhos), 2) as percentual
    FROM empenhos
    GROUP BY orgao
    ORDER BY valor_total DESC
""").fetchdf()

fig_treemap = px.treemap(
    treemap_data,
    labels='orgao',
    values='valor_total',
    color='percentual',
    color_continuous_scale='RdYlGn_r',
    title="🎯 Treemap: Valor por Órgão (% do total)"
)

fig_treemap.write_html("treemap_empenhos.html")
print("✅ Treemap salvo: treemap_empenhos.html")


# ==================== 4. GRÁFICO DE SÉRIE TEMPORAL COM BANDAS ====================
print("\nGerando série temporal com bandas de confiança...")

temporal_detalhado = con.execute("""
    SELECT 
        DATE_TRUNC('week', data_empenho) as semana,
        SUM(valor) as valor_total,
        COUNT(*) as quantidade
    FROM empenhos
    WHERE data_empenho IS NOT NULL
    GROUP BY DATE_TRUNC('week', data_empenho)
    ORDER BY semana
""").fetchdf()

# Calcular média móvel
temporal_detalhado['media_movel_4sem'] = temporal_detalhado['valor_total'].rolling(window=4, center=True).mean()

fig_temporal = go.Figure()

# Área de confiança
fig_temporal.add_trace(go.Scatter(
    x=temporal_detalhado['semana'],
    y=temporal_detalhado['valor_total']/1e6,
    fill='tozeroy',
    name='Valor Semanal',
    line=dict(color='rgba(0,100,80,0)'),
    fillcolor='rgba(0,100,80,0.2)'
))

# Linha de média móvel
fig_temporal.add_trace(go.Scatter(
    x=temporal_detalhado['semana'],
    y=temporal_detalhado['media_movel_4sem']/1e6,
    name='Média Móvel (4 semanas)',
    line=dict(color='#E63946', width=3, dash='dash')
))

fig_temporal.update_layout(
    title="📈 Série Temporal com Média Móvel",
    xaxis_title="Data",
    yaxis_title="Valor (Milhões R$)",
    hovermode='x unified',
    template="plotly_white",
    height=600
)

fig_temporal.write_html("serie_temporal_empenhos.html")
print("✅ Série Temporal salvo: serie_temporal_empenhos.html")


# ==================== 5. COMPARATIVO MULTI-DIMENSÕES ====================
print("\nGerando comparativo multi-dimensões...")

comparativo = con.execute("""
    SELECT 
        YEAR(data_empenho) as ano,
        MONTH(data_empenho) as mes,
        orgao,
        SUM(valor) as valor_total
    FROM empenhos
    WHERE YEAR(data_empenho) IS NOT NULL
        AND MONTH(data_empenho) IS NOT NULL
        AND orgao IN (
            SELECT orgao 
            FROM empenhos 
            GROUP BY orgao 
            ORDER BY SUM(valor) DESC 
            LIMIT 5
        )
    GROUP BY YEAR(data_empenho), MONTH(data_empenho), orgao
    ORDER BY ano DESC, mes, orgao
""").fetchdf()

mes_map = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun',
           7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}
comparativo['mes_nome'] = comparativo['mes'].map(mes_map)

fig_comparativo = px.line(
    comparativo,
    x='mes_nome',
    y='valor_total',
    color='orgao',
    facet_col='ano',
    title="🔄 Top 5 Órgãos: Evolução Mensal",
    labels={'valor_total': 'Valor (R$)', 'mes_nome': 'Mês'},
    markers=True
)

fig_comparativo.update_layout(height=500)
fig_comparativo.write_html("comparativo_orgaos.html")
print("✅ Comparativo salvo: comparativo_orgaos.html")


# ==================== 6. HEATMAP INTERATIVO ====================
print("\nGerando Heatmap...")

heatmap_data = con.execute("""
    SELECT 
        MONTH(data_empenho) as mes,
        YEAR(data_empenho) as ano,
        COUNT(*) as quantidade
    FROM empenhos
    WHERE MONTH(data_empenho) IS NOT NULL 
        AND YEAR(data_empenho) IS NOT NULL
    GROUP BY MONTH(data_empenho), YEAR(data_empenho)
""").fetchdf()

# Pivot para heatmap
pivot = heatmap_data.pivot(index='mes', columns='ano', values='quantidade').fillna(0)

mes_labels = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 
              'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

fig_heatmap = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns,
    y=[mes_labels[i-1] for i in pivot.index],
    colorscale='YlOrRd',
    text=pivot.values,
    texttemplate='%{text:.0f}',
    textfont={"size": 10}
))

fig_heatmap.update_layout(
    title="🔥 Heatmap: Quantidade de Empenhos",
    xaxis_title="Ano",
    yaxis_title="Mês",
    height=500
)

fig_heatmap.write_html("heatmap_empenhos.html")
print("✅ Heatmap salvo: heatmap_empenhos.html")


# ==================== 7. SCATTER PLOT - FREQUÊNCIA vs. VALOR ====================
print("\nGerando Scatter Plot...")

scatter_data = con.execute("""
    SELECT 
        orgao,
        COUNT(*) as frequencia,
        SUM(valor) as valor_total,
        AVG(valor) as valor_medio
    FROM empenhos
    GROUP BY orgao
    HAVING COUNT(*) > 10
    ORDER BY valor_total DESC
""").fetchdf()

fig_scatter = px.scatter(
    scatter_data,
    x='frequencia',
    y='valor_total',
    size='valor_medio',
    color='valor_total',
    hover_name='orgao',
    title="💫 Frequência vs. Valor Total (tamanho = valor médio)",
    labels={'frequencia': 'Quantidade de Empenhos', 'valor_total': 'Valor Total (R$)'},
    color_continuous_scale='Viridis'
)

fig_scatter.update_layout(height=600)
fig_scatter.write_html("scatter_empenhos.html")
print("✅ Scatter Plot salvo: scatter_empenhos.html")


# ==================== RESUMO FINAL ====================
print("\n" + "="*60)
print("✨ VISUALIZAÇÕES GERADAS COM SUCESSO!")
print("="*60)
print("\n📊 Arquivos HTML criados:")
print("  1. dashboard_empenhos.html - Dashboard executivo completo")
print("  2. sunburst_empenhos.html - Hierarquia visual")
print("  3. treemap_empenhos.html - Proporções por órgão")
print("  4. serie_temporal_empenhos.html - Série temporal com média móvel")
print("  5. comparativo_orgaos.html - Comparativo multi-dimensões")
print("  6. heatmap_empenhos.html - Heatmap mês vs. ano")
print("  7. scatter_empenhos.html - Scatter: frequência vs. valor")

print("\n💡 Dica: Abra os arquivos .html em seu navegador!")
print("   Todos têm interatividade: zoom, pan, hover, etc.")

con.close()

Gerando dashboard executivo...
✅ Dashboard salvo: dashboard_empenhos.html

Gerando visualização em Sunburst...
✅ Sunburst salvo: sunburst_empenhos.html

Gerando Treemap...
✅ Treemap salvo: treemap_empenhos.html

Gerando série temporal com bandas de confiança...
✅ Série Temporal salvo: serie_temporal_empenhos.html

Gerando comparativo multi-dimensões...
✅ Comparativo salvo: comparativo_orgaos.html

Gerando Heatmap...
✅ Heatmap salvo: heatmap_empenhos.html

Gerando Scatter Plot...
✅ Scatter Plot salvo: scatter_empenhos.html

✨ VISUALIZAÇÕES GERADAS COM SUCESSO!

📊 Arquivos HTML criados:
  1. dashboard_empenhos.html - Dashboard executivo completo
  2. sunburst_empenhos.html - Hierarquia visual
  3. treemap_empenhos.html - Proporções por órgão
  4. serie_temporal_empenhos.html - Série temporal com média móvel
  5. comparativo_orgaos.html - Comparativo multi-dimensões
  6. heatmap_empenhos.html - Heatmap mês vs. ano
  7. scatter_empenhos.html - Scatter: frequência vs. valor

💡 Dica: Abra os

In [19]:
import plotly.offline as pyo

_plot_original = pyo.plot


def plot_with_div_id(fig, *args, div_id=None, **kwargs):
    if div_id is None:
        return _plot_original(fig, *args, **kwargs)

    include_plotlyjs = kwargs.pop('include_plotlyjs', False)
    config = kwargs.pop('config', None)
    return fig.to_html(
        full_html=False,
        include_plotlyjs=include_plotlyjs,
        config=config,
        div_id=div_id,
    )


pyo.plot = plot_with_div_id

In [20]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.io import to_html
import plotly.offline as pyo

# ==================== CONFIGURAÇÃO ====================
con = duckdb.connect('database.duckdb')

print("📊 Gerando todas as visualizações...")

# Container para armazenar os HTMLs dos gráficos
graficos = {}


# ==================== 1. DASHBOARD EXECUTIVO ====================
print("  → Dashboard executivo...")

kpis = con.execute("""
    SELECT 
        COUNT(*) as total_empenhos,
        SUM(valor) as valor_total,
        ROUND(AVG(valor), 2) as valor_medio,
        COUNT(DISTINCT orgao) as total_orgaos,
        MAX(data_empenho) as ultima_atualizacao
    FROM empenhos
""").fetchdf()

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "Evolução Temporal", "Top 10 Órgãos",
        "Distribuição por Tipo", "Quartis de Valores",
        "Empenhos por Mês", "Concentração (Pareto)"
    ),
    specs=[
        [{"type": "scatter"}, {"type": "bar"}],
        [{"type": "pie"}, {"type": "box"}],
        [{"type": "bar"}, {"type": "scatter"}]
    ]
)

# 1.1 Evolução Temporal
temporal = con.execute("""
    SELECT DATE_TRUNC('month', data_empenho) as periodo,
           SUM(valor) as valor_total
    FROM empenhos
    WHERE data_empenho IS NOT NULL
    GROUP BY DATE_TRUNC('month', data_empenho)
    ORDER BY periodo
""").fetchdf()

fig.add_trace(
    go.Scatter(
        x=temporal['periodo'], y=temporal['valor_total']/1e6,
        mode='lines+markers', name='Valor Total',
        line=dict(color='#2E86AB', width=3), marker=dict(size=5)
    ), row=1, col=1
)

# 1.2 Top 10 Órgãos
top_orgaos = con.execute("""
    SELECT orgao, SUM(valor) as valor_total
    FROM empenhos GROUP BY orgao
    ORDER BY valor_total DESC LIMIT 10
""").fetchdf()

fig.add_trace(
    go.Bar(
        y=top_orgaos['orgao'], x=top_orgaos['valor_total']/1e6,
        orientation='h', name='Órgãos', marker=dict(color='#A23B72')
    ), row=1, col=2
)

# 1.3 Distribuição por Tipo
dist_tipo = con.execute("""
    SELECT tipo, COUNT(*) as quantidade
    FROM empenhos GROUP BY tipo ORDER BY quantidade DESC
""").fetchdf()

fig.add_trace(
    go.Pie(labels=dist_tipo['tipo'], values=dist_tipo['quantidade'], name='Tipo'),
    row=2, col=1
)

# 1.4 Box plot
valores = con.execute("SELECT valor FROM empenhos WHERE valor > 0").fetchall()
fig.add_trace(
    go.Box(
        y=[v[0] for v in valores], name='Distribuição',
        marker=dict(color='#F18F01')
    ), row=2, col=2
)

# 1.5 Empenhos por Mês
empenhos_mes = con.execute("""
    SELECT MONTH(data_empenho) as mes, COUNT(*) as quantidade
    FROM empenhos WHERE MONTH(data_empenho) IS NOT NULL
    GROUP BY MONTH(data_empenho) ORDER BY mes
""").fetchdf()

mes_nomes = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

fig.add_trace(
    go.Bar(
        x=[mes_nomes[int(m)-1] for m in empenhos_mes['mes']],
        y=empenhos_mes['quantidade'], name='Quantidade',
        marker=dict(color='#06A77D')
    ), row=3, col=1
)

# 1.6 Pareto
pareto = con.execute("""
    WITH ranking AS (
        SELECT ROW_NUMBER() OVER (ORDER BY SUM(valor) DESC) as rank,
               orgao, SUM(valor) as valor_total,
               SUM(SUM(valor)) OVER (ORDER BY SUM(valor) DESC) as acumulado,
               SUM(SUM(valor)) OVER () as total
        FROM empenhos GROUP BY orgao
    )
    SELECT rank, 100.0 * acumulado / total as percentual_acumulado
    FROM ranking LIMIT 30
""").fetchdf()

fig.add_trace(
    go.Scatter(
        x=pareto['rank'], y=pareto['percentual_acumulado'],
        mode='lines+markers', name='Acumulado %',
        line=dict(color='#D62828', width=3), marker=dict(size=6)
    ), row=3, col=2
)

fig.update_xaxes(title_text="Data", row=1, col=1)
fig.update_yaxes(title_text="Valor (Milhões R$)", row=1, col=1)
fig.update_xaxes(title_text="Valor (Milhões R$)", row=1, col=2)
fig.update_xaxes(title_text="Quartis", row=2, col=2)
fig.update_yaxes(title_text="Valores (log scale)", type="log", row=2, col=2)
fig.update_xaxes(title_text="Mês", row=3, col=1)
fig.update_xaxes(title_text="Rank de Órgão", row=3, col=2)
fig.update_yaxes(title_text="Acumulado %", row=3, col=2)

fig.update_layout(
    height=1200, width=None, autosize=True,
    title_text="📊 Dashboard Executivo - Empenhos",
    showlegend=False, template="plotly_white"
)

graficos['dashboard'] = fig


# ==================== 2. SUNBURST ====================
print("  → Sunburst...")

hierarchia = con.execute("""
    SELECT COALESCE(tipo, 'Sem Tipo') as nivel_1,
           COALESCE(orgao, 'Sem Órgão') as nivel_2,
           COUNT(*) as quantidade, SUM(valor) as valor_total
    FROM empenhos
    GROUP BY tipo, orgao
    ORDER BY valor_total DESC LIMIT 50
""").fetchdf()

# Construir estrutura correta para sunburst
labels = ['Total']
parents = ['']
values = [hierarchia['valor_total'].sum()]

for tipo in hierarchia['nivel_1'].unique():
    labels.append(tipo)
    parents.append('Total')
    values.append(hierarchia[hierarchia['nivel_1'] == tipo]['valor_total'].sum())

for _, row in hierarchia.iterrows():
    labels.append(f"{row['nivel_1']} - {row['nivel_2']}")
    parents.append(row['nivel_1'])
    values.append(row['valor_total'])

fig_sunburst = go.Figure(go.Sunburst(
    labels=labels, parents=parents, values=values,
    branchvalues='total',
    marker=dict(colorscale='Viridis')
))
fig_sunburst.update_layout(
    title="🌞 Hierarquia: Tipo → Órgão",
    height=600, autosize=True
)
graficos['sunburst'] = fig_sunburst


# ==================== 3. TREEMAP ====================
print("  → Treemap...")

treemap_data = con.execute("""
    SELECT orgao, COUNT(*) as quantidade, SUM(valor) as valor_total,
           ROUND(100.0 * SUM(valor) / (SELECT SUM(valor) FROM empenhos), 2) as percentual
    FROM empenhos GROUP BY orgao ORDER BY valor_total DESC
""").fetchdf()

fig_treemap = px.treemap(
    treemap_data, path=['orgao'], values='valor_total',
    color='percentual', color_continuous_scale='RdYlGn_r',
    title="🎯 Treemap: Valor por Órgão (% do total)"
)
fig_treemap.update_layout(height=600, autosize=True)
graficos['treemap'] = fig_treemap


# ==================== 4. SÉRIE TEMPORAL ====================
print("  → Série temporal...")

temporal_detalhado = con.execute("""
    SELECT DATE_TRUNC('week', data_empenho) as semana,
           SUM(valor) as valor_total, COUNT(*) as quantidade
    FROM empenhos WHERE data_empenho IS NOT NULL
    GROUP BY DATE_TRUNC('week', data_empenho) ORDER BY semana
""").fetchdf()

temporal_detalhado['media_movel_4sem'] = temporal_detalhado['valor_total'].rolling(window=4, center=True).mean()

fig_temporal = go.Figure()
fig_temporal.add_trace(go.Scatter(
    x=temporal_detalhado['semana'], y=temporal_detalhado['valor_total']/1e6,
    fill='tozeroy', name='Valor Semanal',
    line=dict(color='rgba(0,100,80,0)'), fillcolor='rgba(0,100,80,0.2)'
))
fig_temporal.add_trace(go.Scatter(
    x=temporal_detalhado['semana'], y=temporal_detalhado['media_movel_4sem']/1e6,
    name='Média Móvel (4 semanas)',
    line=dict(color='#E63946', width=3, dash='dash')
))
fig_temporal.update_layout(
    title="📈 Série Temporal com Média Móvel",
    xaxis_title="Data", yaxis_title="Valor (Milhões R$)",
    hovermode='x unified', template="plotly_white",
    height=500, autosize=True
)
graficos['temporal'] = fig_temporal


# ==================== 5. COMPARATIVO ====================
print("  → Comparativo multi-dimensões...")

comparativo = con.execute("""
    SELECT YEAR(data_empenho) as ano, MONTH(data_empenho) as mes,
           orgao, SUM(valor) as valor_total
    FROM empenhos
    WHERE YEAR(data_empenho) IS NOT NULL AND MONTH(data_empenho) IS NOT NULL
      AND orgao IN (SELECT orgao FROM empenhos GROUP BY orgao ORDER BY SUM(valor) DESC LIMIT 5)
    GROUP BY YEAR(data_empenho), MONTH(data_empenho), orgao
    ORDER BY ano DESC, mes, orgao
""").fetchdf()

mes_map = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
           7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}
comparativo['mes_nome'] = comparativo['mes'].map(mes_map)

fig_comparativo = px.line(
    comparativo, x='mes_nome', y='valor_total', color='orgao',
    facet_col='ano', title="🔄 Top 5 Órgãos: Evolução Mensal",
    labels={'valor_total':'Valor (R$)','mes_nome':'Mês'}, markers=True
)
fig_comparativo.update_layout(height=500, autosize=True)
graficos['comparativo'] = fig_comparativo


# ==================== 6. HEATMAP ====================
print("  → Heatmap...")

heatmap_data = con.execute("""
    SELECT MONTH(data_empenho) as mes, YEAR(data_empenho) as ano,
           COUNT(*) as quantidade
    FROM empenhos
    WHERE MONTH(data_empenho) IS NOT NULL AND YEAR(data_empenho) IS NOT NULL
    GROUP BY MONTH(data_empenho), YEAR(data_empenho)
""").fetchdf()

pivot = heatmap_data.pivot(index='mes', columns='ano', values='quantidade').fillna(0)

fig_heatmap = go.Figure(data=go.Heatmap(
    z=pivot.values, x=pivot.columns,
    y=[mes_nomes[i-1] for i in pivot.index],
    colorscale='YlOrRd', text=pivot.values,
    texttemplate='%{text:.0f}', textfont={"size": 10}
))
fig_heatmap.update_layout(
    title="🔥 Heatmap: Quantidade de Empenhos",
    xaxis_title="Ano", yaxis_title="Mês",
    height=500, autosize=True
)
graficos['heatmap'] = fig_heatmap


# ==================== 7. SCATTER ====================
print("  → Scatter plot...")

scatter_data = con.execute("""
    SELECT orgao, COUNT(*) as frequencia, SUM(valor) as valor_total,
           AVG(valor) as valor_medio
    FROM empenhos GROUP BY orgao HAVING COUNT(*) > 10
    ORDER BY valor_total DESC
""").fetchdf()

fig_scatter = px.scatter(
    scatter_data, x='frequencia', y='valor_total',
    size='valor_medio', color='valor_total', hover_name='orgao',
    title="💫 Frequência vs. Valor Total (tamanho = valor médio)",
    labels={'frequencia':'Quantidade de Empenhos','valor_total':'Valor Total (R$)'},
    color_continuous_scale='Viridis'
)
fig_scatter.update_layout(height=500, autosize=True)
graficos['scatter'] = fig_scatter


# ==================== KPIs PARA O TOPO ====================
kpi_dict = kpis.iloc[0].to_dict()

# ==================== GERAÇÃO DO HTML ÚNICO ====================
print("\n🎨 Montando HTML único...")

# Gera o plotly.js uma vez (embutido)
plotly_js = '<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>'

# Converter cada figura para HTML (sem plotly.js duplicado)
def fig_to_div(fig, div_id):
    return pyo.plot(fig, include_plotlyjs=False, output_type='div', 
                    div_id=div_id, config={'responsive': True})

html_parts = {
    'dashboard':   fig_to_div(graficos['dashboard'],   'fig_dashboard'),
    'sunburst':    fig_to_div(graficos['sunburst'],    'fig_sunburst'),
    'treemap':     fig_to_div(graficos['treemap'],     'fig_treemap'),
    'temporal':    fig_to_div(graficos['temporal'],    'fig_temporal'),
    'comparativo': fig_to_div(graficos['comparativo'], 'fig_comparativo'),
    'heatmap':     fig_to_div(graficos['heatmap'],     'fig_heatmap'),
    'scatter':     fig_to_div(graficos['scatter'],     'fig_scatter'),
}

# Formata KPIs
def fmt_money(v):
    try:
        return f"R$ {float(v):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except:
        return "R$ 0,00"

html_final = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>📊 Dashboard de Empenhos</title>
{plotly_js}
<style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
        font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
        background: #f4f6f9;
        color: #2c3e50;
        line-height: 1.6;
    }}
    header {{
        background: linear-gradient(135deg, #2E86AB 0%, #A23B72 100%);
        color: white;
        padding: 30px 40px;
        box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    }}
    header h1 {{ font-size: 2rem; margin-bottom: 6px; }}
    header p  {{ opacity: 0.9; font-size: 0.95rem; }}

    .kpi-grid {{
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
        gap: 18px;
        padding: 24px 40px;
        margin-top: -30px;
        position: relative;
        z-index: 10;
    }}
    .kpi {{
        background: white;
        border-radius: 12px;
        padding: 20px;
        box-shadow: 0 4px 14px rgba(0,0,0,0.08);
        border-left: 5px solid #2E86AB;
        transition: transform .2s ease;
    }}
    .kpi:hover {{ transform: translateY(-4px); box-shadow: 0 8px 20px rgba(0,0,0,0.12); }}
    .kpi .label {{ font-size: 0.78rem; color: #7f8c8d; text-transform: uppercase; letter-spacing: .6px; }}
    .kpi .value {{ font-size: 1.5rem; font-weight: 700; color: #2c3e50; margin-top: 6px; }}
    .kpi:nth-child(2) {{ border-left-color: #06A77D; }}
    .kpi:nth-child(3) {{ border-left-color: #F18F01; }}
    .kpi:nth-child(4) {{ border-left-color: #A23B72; }}
    .kpi:nth-child(5) {{ border-left-color: #D62828; }}

    nav.tabs {{
        display: flex; flex-wrap: wrap; gap: 8px;
        padding: 0 40px 20px;
    }}
    nav.tabs button {{
        background: white; border: 1px solid #dfe6ed;
        padding: 9px 18px; border-radius: 25px;
        cursor: pointer; font-size: 0.88rem;
        color: #2c3e50; transition: all .2s;
    }}
    nav.tabs button:hover {{ background: #2E86AB; color: white; border-color: #2E86AB; }}
    nav.tabs button.active {{ background: #2E86AB; color: white; border-color: #2E86AB; }}

    .section {{
        display: none;
        padding: 0 40px 40px;
        animation: fadeIn .35s ease;
    }}
    .section.active {{ display: block; }}
    @keyframes fadeIn {{ from {{ opacity: 0; transform: translateY(8px); }} to {{ opacity: 1; transform: none; }} }}

    .card {{
        background: white;
        border-radius: 12px;
        padding: 15px;
        box-shadow: 0 2px 10px rgba(0,0,0,0.06);
        margin-bottom: 24px;
    }}

    footer {{
        text-align: center;
        padding: 24px;
        color: #95a5a6;
        font-size: 0.85rem;
    }}

    @media (max-width: 768px) {{
        header, nav.tabs, .section, .kpi-grid {{ padding-left: 15px; padding-right: 15px; }}
        header h1 {{ font-size: 1.4rem; }}
        .kpi .value {{ font-size: 1.15rem; }}
    }}
</style>
</head>
<body>

<header>
    <h1>📊 Dashboard Executivo — Empenhos</h1>
    <p>Análise visual consolidada • Atualizado em {kpi_dict.get('ultima_atualizacao', '—')}</p>
</header>

<section class="kpi-grid">
    <div class="kpi"><div class="label">Total de Empenhos</div>
        <div class="value">{int(kpi_dict.get('total_empenhos', 0)):,}</div></div>
    <div class="kpi"><div class="label">Valor Total</div>
        <div class="value">{fmt_money(kpi_dict.get('valor_total', 0))}</div></div>
    <div class="kpi"><div class="label">Valor Médio</div>
        <div class="value">{fmt_money(kpi_dict.get('valor_medio', 0))}</div></div>
    <div class="kpi"><div class="label">Órgãos Distintos</div>
        <div class="value">{int(kpi_dict.get('total_orgaos', 0))}</div></div>
    <div class="kpi"><div class="label">Última Atualização</div>
        <div class="value">{str(kpi_dict.get('ultima_atualizacao', '—'))[:10]}</div></div>
</section>

<nav class="tabs">
    <button class="active" data-target="sec-dashboard">📊 Dashboard</button>
    <button data-target="sec-sunburst">🌞 Sunburst</button>
    <button data-target="sec-treemap">🎯 Treemap</button>
    <button data-target="sec-temporal">📈 Série Temporal</button>
    <button data-target="sec-comparativo">🔄 Comparativo</button>
    <button data-target="sec-heatmap">🔥 Heatmap</button>
    <button data-target="sec-scatter">💫 Scatter</button>
</nav>

<div class="section active" id="sec-dashboard">
    <div class="card">{html_parts['dashboard']}</div>
</div>
<div class="section" id="sec-sunburst">
    <div class="card">{html_parts['sunburst']}</div>
</div>
<div class="section" id="sec-treemap">
    <div class="card">{html_parts['treemap']}</div>
</div>
<div class="section" id="sec-temporal">
    <div class="card">{html_parts['temporal']}</div>
</div>
<div class="section" id="sec-comparativo">
    <div class="card">{html_parts['comparativo']}</div>
</div>
<div class="section" id="sec-heatmap">
    <div class="card">{html_parts['heatmap']}</div>
</div>
<div class="section" id="sec-scatter">
    <div class="card">{html_parts['scatter']}</div>
</div>

<footer>
    ✨ Gerado automaticamente com Python + DuckDB + Plotly
</footer>

<script>
    // Navegação por abas
    const buttons = document.querySelectorAll('nav.tabs button');
    const sections = document.querySelectorAll('.section');

    buttons.forEach(btn => {{
        btn.addEventListener('click', () => {{
            buttons.forEach(b => b.classList.remove('active'));
            sections.forEach(s => s.classList.remove('active'));
            btn.classList.add('active');
            document.getElementById(btn.dataset.target).classList.add('active');

            // Redimensiona gráficos Plotly (importante após mudança de visibilidade)
            setTimeout(() => {{
                document.querySelectorAll('.section.active .js-plotly-plot').forEach(el => {{
                    try {{ Plotly.Plots.resize(el); }} catch(e) {{}}
                }});
            }}, 60);
        }});
    }});

    // Redimensiona ao trocar de tela
    window.addEventListener('resize', () => {{
        document.querySelectorAll('.js-plotly-plot').forEach(el => {{
            try {{ Plotly.Plots.resize(el); }} catch(e) {{}}
        }});
    }});
</script>

</body>
</html>
"""

# Salvar arquivo único
with open("dashboard_completo.html", "w", encoding="utf-8") as f:
    f.write(html_final)

print("\n" + "="*60)
print("✨ SUCESSO! Arquivo único gerado.")
print("="*60)
print("📄 Arquivo: dashboard_completo.html")
print("\n💡 Contém TODAS as 7 visualizações + KPIs + navegação por abas.")
print("   Abra em qualquer navegador moderno. 100% offline e interativo.")

con.close()

📊 Gerando todas as visualizações...
  → Dashboard executivo...
  → Sunburst...
  → Treemap...
  → Série temporal...
  → Comparativo multi-dimensões...
  → Heatmap...
  → Scatter plot...

🎨 Montando HTML único...

✨ SUCESSO! Arquivo único gerado.
📄 Arquivo: dashboard_completo.html

💡 Contém TODAS as 7 visualizações + KPIs + navegação por abas.
   Abra em qualquer navegador moderno. 100% offline e interativo.
